# Prefix search

現在 token（suffix）を固定し、語彙中のすべての過去 token（prefix）について $T^{ee}$（LN あり）と OpenWebText での bigram 数を並べる。

`bash scripts/compute_data.sh frequency` で OpenWebText の頻度（`outputs/corpus-tools/` の下）を作っておく。

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import polars as pl
from feature_extractor.models import load_tokenizer

from lm_detokenization.analysis.detokenization import TokenAffinity, roc
from lm_detokenization.data.corpus import counts_dir, load_bigram_counts
from lm_detokenization.weights import load_layer0_weights

ROOT = Path.cwd().parent  # notebooks/ -> repository root
MODEL_NAME = "gpt2"

tokenizer = load_tokenizer(MODEL_NAME)
affinity = TokenAffinity.from_weights(load_layer0_weights(MODEL_NAME))
bigrams = load_bigram_counts(ROOT / counts_dir(MODEL_NAME)).tocsc()
# token as written in the paper, with "_" for a leading space
TOKENS = [t.replace("Ġ", "_") for t in tokenizer.convert_ids_to_tokens(range(len(tokenizer)))]

In [ ]:
def search(word: str) -> pl.DataFrame:
    """T^ee and bigram count of every (prefix, suffix) pair for every head,
    with the last token of `word` as the suffix."""
    suffix_id = tokenizer.encode(word, add_special_tokens=False)[-1]
    print(f"suffix: {TOKENS[suffix_id]!r} ({suffix_id})")
    scores = affinity.suffix_scores([suffix_id])[0].numpy()  # [head, prefix]
    num_heads, vocab_size = scores.shape
    counts = bigrams[:, suffix_id].toarray().ravel()
    return pl.DataFrame(
        {
            "prefix_id": np.tile(np.arange(vocab_size), num_heads),
            "suffix_id": suffix_id,
            "head": np.repeat(np.arange(num_heads), vocab_size),
            "score": scores.ravel(),
            "count": np.tile(counts, num_heads),
        }
    ).with_columns(
        pl.col("prefix_id")
        .map_elements(lambda i: TOKENS[i] + TOKENS[suffix_id].lstrip("_"), return_dtype=pl.String)
        .alias("detokenization")
    )

In [ ]:
df = search(" sapiens")
df

In [ ]:
head = 7
df_head = df.filter(pl.col("head") == head)
print(f"Head: {head}")
with pl.Config(tbl_rows=10):
    # by T^ee
    display(df_head.sort("score", descending=True).head(10))
    # by T^ee, among bigrams that occur more than once
    display(df_head.filter(pl.col("count") > 1).sort("score", descending=True).head(10))
    # by bigram count
    display(df_head.sort("count", descending=True).head(10))

In [ ]:
# ROC of this head (bigram counts as weights of the positives)
fpr, tpr, auc = roc(affinity, bigrams, int(df_head["suffix_id"][0]), head)
fig, ax = plt.subplots(figsize=(3, 3))
ax.plot(fpr, tpr)
ax.fill_between(fpr, tpr, alpha=0.2)
ax.plot([0, 1], [0, 1], ls="--", c=".3", lw=0.8)
ax.set(xlim=(0, 1), ylim=(0, 1), xlabel="False Positive Rate", ylabel="True Positive Rate",
       title=f"{TOKENS[int(df_head['suffix_id'][0])]!r}, head {head}: AUC {auc:.2f}")